# Figure 1F, G, H — Steroid Dose Violin Plots

Violin plots of **log2(post/pre)** prednisone-equivalent steroid exposure by AE grade.

**Methodology:**
- Pre-treatment window: 60 days before AE onset (or timeline midpoint for Grade 0)
- Post-treatment window: 60 days after AE onset (or timeline midpoint for Grade 0)
- **Filter to ICI+ patients only** (same as original pipeline)
- Filter to patients with ANY steroid exposure (pre or post > 0)
- Uses **new LOT files** from `OneDrive_1_8-7-2026` (pre-merged with ICI status)

**Prerequisites:** Run cache generation script first, or use pre-computed `steroid_ratio_cache_LOT.pkl`

**Outputs:** PDF, PNG, and CSV in `results/main/` (1F–H). Unpublished extras for liver toxicity, hypothyroidism, and hyperthyroidism go to `results/extra_steroid/`.

In [ ]:
# ---------------------------------------------------------------------------
# IMPORTS
# ---------------------------------------------------------------------------
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import MaxNLocator
from scipy.stats import mannwhitneyu, norm

import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    print(f"Warning: Arial not found, using {_arial_path}")
else:
    print(f"Arial resolved to: {_arial_path}")

In [ ]:
# ---------------------------------------------------------------------------
# FILE PATHS
# ---------------------------------------------------------------------------
ROOT = Path("..").resolve()
FIGURES = ROOT.parent.parent
DATA = FIGURES / "figures_data" / "figure 1" / "data"
RESULTS = ROOT / "results"
FIG_DIR = RESULTS / "main"
EXTRA_STEROID_DIR = RESULTS / "extra_steroid"
FIG_DIR.mkdir(parents=True, exist_ok=True)
EXTRA_STEROID_DIR.mkdir(parents=True, exist_ok=True)

CACHE_PATH = DATA / "steroid_ratio_cache_LOT.pkl"
assert CACHE_PATH.exists(), f"Missing: {CACHE_PATH.name} (run generate_steroid_cache.py first)"

print(f"Cache: {CACHE_PATH.name}")
print(f"Output folder: {FIG_DIR.name}")
print(f"Extra steroid folder: {EXTRA_STEROID_DIR.name}")

In [ ]:
# ---------------------------------------------------------------------------
# LOAD CACHE AND FILTER (ICI+ patients with steroid exposure)
# ---------------------------------------------------------------------------
# Cache is a dict of numpy arrays (not a pandas DataFrame) so it loads
# across pandas 1/2/3. Rebuild the frame here.
with open(CACHE_PATH, 'rb') as f:
    _raw = pickle.load(f)

if isinstance(_raw, dict) and _raw.get("_compat") == "numpy_columns_v1":
    _data = {}
    for _col in _raw["columns"]:
        _arr = _raw["arrays"][_col]
        _kind = _raw["kinds"][_col]
        if _kind == "datetime_ns":
            _data[_col] = pd.to_datetime(_arr, unit="ns")
        else:
            _data[_col] = _arr
    full_df = pd.DataFrame(_data, columns=_raw["columns"])
else:
    full_df = _raw
    if "ref_date" in full_df.columns:
        full_df["ref_date"] = pd.to_datetime(full_df["ref_date"])

print(f"Loaded {len(full_df):,} patient-toxicity records")
print(f"ICI+ patients: {full_df['ici_at_ref'].sum():,} ({100*full_df['ici_at_ref'].mean():.1f}%)")

# Step 1: Filter to ICI+ patients (same as original pipeline)
ici_df = full_df[full_df['ici_at_ref'] == True].copy()
print(f"\nAfter ICI+ filter: {len(ici_df):,} records")

# Step 2: Filter to patients with ANY steroid exposure (pre or post > 0)
results_df = ici_df[(ici_df['pre_steroid'] > 0) | (ici_df['post_steroid'] > 0)].copy()
print(f"After steroid filter: {len(results_df):,} records with steroid exposure ({100*len(results_df)/len(ici_df):.1f}% of ICI+)")

print(f"\nAvailable toxicities (ICI+ with steroid):")
for tox in sorted(results_df['toxicity'].unique()):
    n = len(results_df[results_df['toxicity'] == tox])
    print(f"  {tox}: {n:,} patients")

In [ ]:
# ---------------------------------------------------------------------------
# PLOTTING CONFIG
# ---------------------------------------------------------------------------
MIN_N = 5
GRADE_GRAY = '#95a5a6'

# Linear y-axis, truncated at +/- Y_LIM so the grade-to-grade median step is
# readable. Tails beyond this window (the zero-pre / zero-post clumps near
# +/-16) are omitted from the drawing only -- medians, Mann-Whitney tests and
# the trend test still use every value. process_ae() prints the omitted count.
Y_LIM = 8
MEDIAN_CONNECTOR = False  # join the group medians to show the trend directly
MEDIAN_LABELS = False     # print each group median next to its bar

AE_CONFIGS = [
    {
        "panel": "1F",
        "toxicity": "adrenal_insufficiency",
        "title": "Adrenal Insufficiency",
        "ylabel": "log2(post/pre)\nprednisone equivalent in mg-days",
        "out_pdf": "Steroid_Violin_Adrenal_Insuff_1F.pdf",
        "out_png": "Steroid_Violin_Adrenal_Insuff_1F.png",
        "out_csv": "Steroid_Violin_Adrenal_Insuff_1F_stats.csv",
        "force_groups": {0: [0], 1.5: [1, 2], 3.5: [3, 4]},  # Combine 1-2 and 3-4
    },
    {
        "panel": "1G",
        "toxicity": "colitis",
        "title": "Colitis",
        "ylabel": "log2(post/pre)\nprednisone equivalent in mg-days",
        "out_pdf": "Steroid_Violin_Colitis_1G.pdf",
        "out_png": "Steroid_Violin_Colitis_1G.png",
        "out_csv": "Steroid_Violin_Colitis_1G_stats.csv",
        "force_groups": {0: [0], 1.5: [1, 2], 3.5: [3, 4]},  # Combine 1-2 and 3-4
    },
    {
        "panel": "1H",
        "toxicity": "pneumonitis",
        "title": "Pneumonitis",
        "ylabel": "log2(post/pre)\nprednisone equivalent in mg-days",
        "out_pdf": "Steroid_Violin_Pneumonitis_1H.pdf",
        "out_png": "Steroid_Violin_Pneumonitis_1H.png",
        "out_csv": "Steroid_Violin_Pneumonitis_1H_stats.csv",
        "force_groups": {0: [0], 1.5: [1, 2], 3.5: [3, 4]},  # Combine 1-2 and 3-4
    },
]

EXTRA_AE_CONFIGS = [
    {
        "panel": "extra",
        "toxicity": "liver_toxicity",
        "title": "Liver Toxicity",
        "ylabel": "log2(post/pre)\nprednisone equivalent in mg-days",
        "out_pdf": "Steroid_Violin_Liver_Tox.pdf",
        "out_png": "Steroid_Violin_Liver_Tox.png",
        "out_csv": "Steroid_Violin_Liver_Tox_stats.csv",
        "force_groups": {0: [0], 1.5: [1, 2], 3.5: [3, 4]},
    },
    {
        "panel": "extra",
        "toxicity": "hypothyroidism",
        "title": "Hypothyroidism",
        "ylabel": "log2(post/pre)\nprednisone equivalent in mg-days",
        "out_pdf": "Steroid_Violin_Hypothyroidism.pdf",
        "out_png": "Steroid_Violin_Hypothyroidism.png",
        "out_csv": "Steroid_Violin_Hypothyroidism_stats.csv",
        "force_groups": {0: [0], 1.5: [1, 2], 3.5: [3, 4]},
    },
    {
        "panel": "extra",
        "toxicity": "hyperthyroidism",
        "title": "Hyperthyroidism",
        "ylabel": "log2(post/pre)\nprednisone equivalent in mg-days",
        "out_pdf": "Steroid_Violin_Hyperthyroidism.pdf",
        "out_png": "Steroid_Violin_Hyperthyroidism.png",
        "out_csv": "Steroid_Violin_Hyperthyroidism_stats.csv",
        "force_groups": {0: [0], 1.5: [1, 2], 3.5: [3, 4]},
    },
]

def get_grade_colors(grades):
    """Gray for grade 0; light -> dark rocket_r for grade 1+"""
    cmap = sns.color_palette("rocket_r", as_cmap=True)
    non_zero = sorted(g for g in grades if g != 0)
    n = len(non_zero)
    shades = {}
    if n > 0:
        t_vals = np.linspace(0.25, 0.85, n) if n > 1 else [0.55]
        for g, t in zip(non_zero, t_vals):
            shades[g] = cmap(t)
    return {0: GRADE_GRAY, **shades}

print("Config loaded.")

In [ ]:
# ---------------------------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------------------------
def group_low_count_grades(data_by_grade, min_n=5):
    """Merge grades with fewer than min_n samples with adjacent grades."""
    if not data_by_grade:
        return {}
    numeric = {}
    for k, v in data_by_grade.items():
        try:
            numeric[float(k)] = v
        except (ValueError, TypeError):
            continue
    if not numeric:
        return {}
    grades = sorted(numeric.keys())
    if not any(len(numeric[g]) < min_n for g in grades if g != 0.0):
        return numeric
    grouped, i = {}, 0
    while i < len(grades):
        g = grades[i]
        d = numeric[g]
        if g == 0.0:
            grouped[0.0] = d; i += 1; continue
        if len(d) < min_n and i + 1 < len(grades):
            ng = grades[i + 1]
            grouped[(g + ng) / 2] = np.concatenate([d, numeric[ng]])
            i += 2
        elif len(d) < min_n and i > 0:
            pg = grades[i - 1]
            if pg != 0.0 and pg in grouped:
                grouped[pg] = np.concatenate([grouped[pg], d])
            elif pg != 0.0:
                grouped[(pg + g) / 2] = np.concatenate([numeric[pg], d])
            else:
                grouped[g] = d
            i += 1
        else:
            grouped[g] = d; i += 1
    return grouped

def grade_label(grade):
    if grade == 0.0:
        return 'No AE'
    if grade == int(grade):
        return f'Grade {int(grade)}'
    # For combined grades like 1.5 (Grade 1-2) or 3.5 (Grade 3-4)
    low = int(grade - 0.5)
    high = int(grade + 0.5)
    return f'Grade {low}-{high}'

def jonckheere_terpstra(groups):
    """Jonckheere-Terpstra trend test."""
    k = len(groups)
    n_total = sum(len(g) for g in groups)
    JT = 0
    for i in range(k):
        for j in range(i + 1, k):
            gi, gj = np.asarray(groups[i]), np.asarray(groups[j])
            U = int((gj[:, None] > gi[None, :]).sum()) if len(gi) and len(gj) else 0
            JT += U
    n_i = np.array([len(g) for g in groups])
    mean_JT = (n_total**2 - np.sum(n_i**2)) / 4
    var_JT = (n_total**2 * (2 * n_total + 3) - np.sum(n_i**2 * (2 * n_i + 3))) / 72
    if var_JT <= 0:
        return JT, np.nan, np.nan
    z = (JT - mean_JT) / np.sqrt(var_JT)
    log_p = np.log(2) + norm.logsf(abs(z))
    p = np.exp(log_p) if log_p > -700 else 0.0
    return JT, z, p

print("Helper functions defined.")

In [ ]:
# ---------------------------------------------------------------------------
# MAIN PLOTTING FUNCTION
# ---------------------------------------------------------------------------
def process_ae(cfg, results_df, out_dir=None):
    toxicity, ylabel = cfg["toxicity"], cfg["ylabel"]
    out_dir = Path(out_dir) if out_dir is not None else FIG_DIR
    pdf_path = out_dir / cfg["out_pdf"]
    png_path = out_dir / cfg["out_png"]
    csv_path = out_dir / cfg["out_csv"]

    sub = results_df[results_df['toxicity'] == toxicity].copy()
    print(f"\n=== [{cfg['panel']}] {cfg['title']} -- {len(sub):,} patients with steroid ===")

    # Collect log2_ratio values by grade
    data_by_grade = {}
    raw_by_grade = {}  # post_steroid values for raw stats
    for g in sub['grade'].unique():
        log2_vals = sub.loc[sub['grade'] == g, 'log2_ratio'].dropna().values
        raw_vals = sub.loc[sub['grade'] == g, 'post_steroid'].dropna().values
        if len(log2_vals) > 0:
            data_by_grade[float(g)] = log2_vals
            raw_by_grade[float(g)] = raw_vals

    # Check for forced groupings (e.g., combine Grade 1-2 and 3-4)
    if "force_groups" in cfg:
        grouped = {}
        raw_grouped = {}
        for new_grade, old_grades in cfg["force_groups"].items():
            combined_log2 = np.concatenate([data_by_grade.get(float(g), np.array([])) for g in old_grades])
            combined_raw = np.concatenate([raw_by_grade.get(float(g), np.array([])) for g in old_grades])
            if len(combined_log2) > 0:
                grouped[float(new_grade)] = combined_log2
                raw_grouped[float(new_grade)] = combined_raw
    else:
        grouped = group_low_count_grades(data_by_grade, min_n=MIN_N)
        raw_grouped = group_low_count_grades(raw_by_grade, min_n=MIN_N)
    
    if not grouped:
        print(f"  WARNING: no data -- skipping.")
        return

    grades = sorted(grouped.keys())

    # Adjacent-grade Mann-Whitney p-values
    p_vals = [None]
    for i in range(1, len(grades)):
        a, b = grouped[grades[i - 1]], grouped[grades[i]]
        if len(a) >= 3 and len(b) >= 3:
            try:
                p_vals.append(mannwhitneyu(a, b, alternative='two-sided').pvalue)
            except Exception:
                p_vals.append(None)
        else:
            p_vals.append(None)

    grade_colors = get_grade_colors(grades)

    labels, data_lists, colors = [], [], []
    for g in grades:
        d = grouped[g]
        labels.append(f'{grade_label(g)}\nn={len(d):,}')
        data_lists.append(d)
        colors.append(grade_colors.get(g, GRADE_GRAY))

    jt_stat, jt_z, jt_p = jonckheere_terpstra(data_lists)
    jt_ptxt = 'Trend p<0.001' if jt_p < 0.001 else f'Trend p={jt_p:.3f}'
    print(f"  Jonckheere-Terpstra: JT={jt_stat}, z={jt_z:.3f}, p={jt_p:.4g}")

    # Stats rows for CSV
    stat_rows = []
    for i, g in enumerate(grades):
        d = grouped[g]  # log2 ratio values
        r = raw_grouped[g]  # raw post_steroid values
        stat_rows.append({
            "panel": cfg["panel"], "ae_type": toxicity,
            "lab_test": "prednisone_equivalent",
            "grade": grade_label(g), "n": len(d),
            "log2_median": float(np.median(d)),
            "log2_q1": float(np.percentile(d, 25)),
            "log2_q3": float(np.percentile(d, 75)),
            "raw_median": float(np.median(r)),
            "raw_q1": float(np.percentile(r, 25)),
            "raw_q3": float(np.percentile(r, 75)),
            "raw_mean": float(np.mean(r)),
            "raw_std": float(np.std(r)),
            "p_vs_prev_grade": p_vals[i],
            "trend_JT_stat": jt_stat, "trend_z": jt_z, "trend_p": jt_p,
        })
    
    # Save CSV
    stats_df = pd.DataFrame(stat_rows)
    stats_df.to_csv(csv_path, index=False)
    print(f"  Saved CSV: {csv_path.name}")

    # Violin plot
    fig, ax = plt.subplots(figsize=(2.8, 1.9), constrained_layout=True)
    positions = list(range(len(grades)))

    all_vals = np.concatenate(data_lists)
    n_out = int((np.abs(all_vals) > Y_LIM).sum())
    print(f"  Y view: linear +/-{Y_LIM:g} -- {n_out:,} of {len(all_vals):,} points "
          f"({100*n_out/len(all_vals):.1f}%) outside the visible range")

    parts = ax.violinplot(data_lists, positions=positions, showmeans=False,
                          showmedians=False, showextrema=False, widths=0.6)
    for pc, c in zip(parts['bodies'], colors):
        pc.set_facecolor(c); pc.set_alpha(0.35)
        pc.set_edgecolor(c); pc.set_linewidth(1.2)

    # Lighter, smaller points so they read as a density cloud and stop
    # competing with the median bars for attention.
    np.random.seed(42)
    medians = []
    for pos, d, c in zip(positions, data_lists, colors):
        if len(d) > 0:
            jitter = np.random.normal(pos, 0.05, len(d))
            ax.scatter(jitter, d, alpha=0.18, s=4, color=c,
                       linewidths=0, zorder=3)
            ax.hlines(np.median(d), pos - 0.26, pos + 0.26,
                      colors='black', linewidth=2.2, zorder=6)
            medians.append(float(np.median(d)))
        else:
            medians.append(np.nan)

    if MEDIAN_CONNECTOR and len(positions) > 1:
        ax.plot(positions, medians, color='#b03a2e', linewidth=1.1,
                marker='o', markersize=3, markeredgecolor='white',
                markeredgewidth=0.4, zorder=7)

    if MEDIAN_LABELS:
        for pos, med in zip(positions, medians):
            ax.annotate(f'{med:+.2f}', xy=(pos, med),
                        xytext=(0, 7), textcoords='offset points',
                        ha='center', va='bottom', fontsize=5.5,
                        fontweight='bold', color='#7b241c', zorder=8,
                        bbox=dict(boxstyle='round,pad=0.12', fc='white',
                                  ec='none', alpha=0.75))

    ax.set_ylim(-Y_LIM, Y_LIM)
    ax.set_yticks([-8, -4, 0, 4, 8])

    # Brackets sit just above the axis frame (x in data coords, y in axes
    # fraction) so they cost no vertical space inside the panel.
    trans = ax.get_xaxis_transform()
    for i in range(1, len(grades)):
        if p_vals[i] is None:
            continue
        x1, x2 = positions[i - 1] + 0.07, positions[i] - 0.07
        ax.plot([x1, x1, x2, x2], [1.005, 1.035, 1.035, 1.005], transform=trans,
                color='black', linewidth=0.9, clip_on=False, zorder=9)
        ptxt = 'p<0.001' if p_vals[i] < 0.001 else f'p={p_vals[i]:.3f}'
        ax.text((x1 + x2) / 2, 1.045, ptxt, transform=trans, ha='center',
                va='bottom', fontsize=5, clip_on=False, zorder=9)

    # y=0 means post-window exposure equals pre-window exposure
    ax.axhline(0, color='#7f8c8d', linewidth=0.6, linestyle=':', zorder=1)

    ax.set_ylabel(ylabel, fontsize=7)
    ax.set_title(cfg["title"], fontsize=8, fontweight='bold', pad=18)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, fontsize=6)
    ax.tick_params(axis='y', labelsize=6)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.text(0.02, 0.98, jt_ptxt, transform=ax.transAxes,
            ha='left', va='top', fontsize=5, style='italic',
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none',
                      alpha=0.7))

    # Save PDF
    with PdfPages(str(pdf_path)) as pdf:
        pdf.savefig(fig, dpi=450)
    print(f"  Saved PDF: {pdf_path.name}")
    
    # Save PNG
    fig.savefig(png_path, dpi=450, bbox_inches='tight')
    print(f"  Saved PNG: {png_path.name}")
    
    plt.show()

print("process_ae() defined.")

In [ ]:
# ---------------------------------------------------------------------------
# RUN ALL PANELS
# ---------------------------------------------------------------------------
print("="*80)
print("GENERATING STEROID VIOLIN FIGURES")
print("="*80)

for cfg in AE_CONFIGS:
    process_ae(cfg, results_df, FIG_DIR)

print("\n" + "="*80)
print("GENERATING EXTRA STEROID (unpublished)")
print("="*80)

for cfg in EXTRA_AE_CONFIGS:
    process_ae(cfg, results_df, EXTRA_STEROID_DIR)

print("\n" + "="*80)
print("DONE - published panels in results/main/; extras in results/extra_steroid/")
print("="*80)